# Перебор признаков

In [ ]:
'''
Что известно:
- 222 признака, LightGBM + XGBoost + CatBoost с подбором: 0.726 на кросс-валидации, около 0.72 на Public;
- по словам преподавателя, решение уровня Middle (0.837) построено только на признаках, и даже простые модели
  набирают этот скор; значит, нам не хватает одного-двух сильных признаков;
- у клиентов с 2–3 днями заказов target почти предопределён днём «последний заказ + средний интервал»;
  для 4+ дней такой формулы мы не нашли, хотя перебирали её вручную и только в днях;
- метки train ведут себя не так, как покупки, видимые в самой истории.

Что делаем здесь:
1. массово строим кандидатов «ожидаемый момент следующей покупки»: 4 единицы времени (день, заказ, календарная
   неделя, календарный месяц) x 3 глубины истории x 11 оценок периода x 5 видов признака — около 700 штук;
2. оцениваем каждого в одиночку: AP@20% и AUC внутри групп по числу дней с заказами, особенно 4+;
3. добавляем лучших к 222 признакам (LightGBM) и к 10 простым (логистическая регрессия);
4. на сдвинутых датах считаем метки по нескольким определениям target и ищем то, что воспроизводит train.
'''
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)
data["top_category2"] = data["top_category2"].astype("category")
BASE = [c for c in feats.columns if c != "top_brand"]

cid = "customer_id"
info = cust.set_index(cid)
tr = data[data["part"].eq("train")].reset_index(drop=True)
tr_ids = tr[cid].to_numpy()
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()
slices = np.unique(sl)
nd = tr["n_days"].to_numpy()
GROUPS = {"2-3": (nd >= 2) & (nd <= 3), "4-5": (nd >= 4) & (nd <= 5), "6-9": (nd >= 6) & (nd <= 9), "10-20": (nd >= 10) & (nd <= 20), "21+": nd >= 21}
print(f"базовых признаков: {len(BASE)} | {(time.time() - T0) / 60:.1f} мин")

## Кандидаты

In [ ]:
o_lvl = lines.drop_duplicates([cid, "order_id"])[[cid, "order_date", "d"]]
WEEK0 = pd.Timestamp("2019-01-07")   # понедельник


def week_idx(s):
    return (s - WEEK0).dt.days // 7


def month_idx(s):
    return s.dt.year * 12 + s.dt.month


ref_week = week_idx(info["scoring_date"])
ref_month = month_idx(info["scoring_date"])
days_u = o_lvl[[cid, "d"]].drop_duplicates()
# единица: события клиента со временем t в единицах до опорной точки; дней в единице; окно target (lo, hi] в единицах после опорной точки.
# Для дней и заказов опорная точка — отсечение, окно (14, 14 + длина окна]; для недель и месяцев — дата скоринга, окно — 13 недель или 3 календарных месяца.
UNITS = {
    "день": (days_u.assign(t=days_u["d"].astype(float)), 1.0, (float(GAP_DAYS), None)),
    "заказ": (o_lvl.assign(t=o_lvl["d"].astype(float))[[cid, "d", "t"]], 1.0, (float(GAP_DAYS), None)),
    "неделя": (o_lvl.assign(t=(o_lvl[cid].map(ref_week) - week_idx(o_lvl["order_date"])).astype(float))[[cid, "d", "t"]].drop_duplicates([cid, "t"]), 7.0, (-0.5, 12.5)),
    "месяц": (o_lvl.assign(t=(o_lvl[cid].map(ref_month) - month_idx(o_lvl["order_date"])).astype(float))[[cid, "d", "t"]].drop_duplicates([cid, "t"]), 30.44, (-0.5, 2.5)),
}
HIST = {"вся": 10_000, "год": 365, "полгода": 182}
EST = ["mean", "median", "min", "max", "q25", "q75", "last", "last2", "last3", "ewm", "trim"]


def estimators(E):
    E = E.sort_values([cid, "t"]).reset_index(drop=True)
    g = E.groupby(cid)["t"]
    E["gap"] = g.diff()
    E["rk"] = E.groupby(cid).cumcount()
    gp = E[E["gap"].notna()]
    gg = gp.groupby(cid)["gap"]
    out = pd.DataFrame({"r": g.min(), "n": g.size()})
    out["mean"] = (g.max() - g.min()) / (out["n"] - 1).replace(0, np.nan)
    out["median"] = gg.median()
    out["min"] = gg.min()
    out["max"] = gg.max()
    out["q25"] = gg.quantile(0.25)
    out["q75"] = gg.quantile(0.75)
    out["last"] = gp[gp["rk"] == 1].set_index(cid)["gap"]
    out["last2"] = gp[gp["rk"] <= 2].groupby(cid)["gap"].mean()
    out["last3"] = gp[gp["rk"] <= 3].groupby(cid)["gap"].mean()
    w = 0.7 ** (gp["rk"] - 1)
    out["ewm"] = (gp["gap"] * w).groupby(gp[cid]).sum() / w.groupby(gp[cid]).sum()
    out["trim"] = (gg.sum() - gg.max()) / (gg.count() - 1).replace(0, np.nan)
    return out


cand = {}
for unit, (E_all, scale, (lo_u, hi_u)) in UNITS.items():
    for hname, H in HIST.items():
        est = estimators(E_all[E_all["d"] <= H][[cid, "t"]]).reindex(info.index)
        r = est["r"]
        lo = pd.Series(lo_u, index=info.index, dtype=float)
        hi = (GAP_DAYS + info["win_len"]).astype(float) if hi_u is None else pd.Series(hi_u, index=info.index, dtype=float)
        key = f"{unit}|{hname}"
        cand[f"{key}|n"] = est["n"]
        cand[f"{key}|r"] = r
        for e in EST:
            P = est[e].where(est[e] > 0)
            tn = P - r                                  # ожидаемый момент следующей покупки после опорной точки
            tm = P - np.mod(r, P)                       # первый момент по сетке периода после опорной точки
            pre = f"{key}|{e}"
            cand[f"{pre}|t"] = tn * scale
            cand[f"{pre}|dist"] = ((lo - tn).clip(lower=0) + (tn - hi).clip(lower=0)) * scale
            cand[f"{pre}|in"] = ((tn > lo) & (tn <= hi)).astype("float32").where(P.notna())
            cand[f"{pre}|tm_in"] = ((tm > lo) & (tm <= hi)).astype("float32").where(P.notna())
            cand[f"{pre}|k_in"] = np.floor((r + hi) / P) - np.floor((r + lo) / P)   # сколько раз сетка периода попадает в окно
    print(f"{unit}: готово, {(time.time() - T0) / 60:.1f} мин")
CAND = pd.DataFrame(cand).astype("float32")
print("кандидатов:", CAND.shape[1])

## Оценка кандидатов по одиночке

In [ ]:
def screen(v):
    v = np.asarray(v, dtype=float)
    nan = np.isnan(v)
    row = {"AP@20%": -1.0}
    for sign in (1, -1):
        s = np.where(nan, -np.inf, sign * v)
        ap = float(np.mean([fast_ap(y[sl == q], s[sl == q]) for q in slices]))
        if ap > row["AP@20%"]:
            row = {"AP@20%": ap, "знак": sign}
    row["покрытие"] = 1 - nan.mean()
    aucs = []
    for g, m in GROUPS.items():
        m2 = m & ~nan
        if m2.sum() > 200 and 0 < y[m2].mean() < 1 and np.std(v[m2]) > 0:
            a = roc_auc_score(y[m2], v[m2])
            row[f"AUC {g}"] = max(a, 1 - a)
            if g != "2-3":
                aucs.append(row[f"AUC {g}"])
        else:
            row[f"AUC {g}"] = np.nan
    row["AUC 4+ среднее"] = float(np.mean(aucs)) if aucs else np.nan
    return row


Xc = CAND.reindex(tr_ids).reset_index(drop=True)
rows = {c: screen(Xc[c].to_numpy()) for c in CAND.columns}
for c in ["formula", "days_formula", "recency", "n_days", "tnext_mean_dist", "phit20_median", "recency_to_gap_median", "recency_to_tenure"]:
    rows[f"[база] {c}"] = screen(tr[c].to_numpy())
S = pd.DataFrame.from_dict(rows, orient="index")
S["единица"] = [i.split("|")[0] if "|" in i else "база" for i in S.index]
print(f"оценка: {(time.time() - T0) / 60:.1f} мин")
print("Опорные признаки из текущего набора:")
display(S[S.index.str.startswith("[база]")])
print("Лучшие кандидаты по средней AUC внутри групп с 4+ днями заказов:")
display(S.sort_values("AUC 4+ среднее", ascending=False).head(40))
print("Лучшие кандидаты по AP@20% в одиночку:")
display(S.sort_values("AP@20%", ascending=False).head(25))
print("Лучший кандидат каждой единицы времени:")
display(S.groupby("единица")[["AP@20%", "AUC 2-3", "AUC 4+ среднее"]].max())

top_in = [c for c in S.sort_values("AUC 4+ среднее", ascending=False).index if c.endswith("|in") or c.endswith("|tm_in")][:8]
zr = {}
for c in top_in:
    v = Xc[c].to_numpy()
    for g, m in GROUPS.items():
        zr.setdefault(c, {})[f"{g}: в окне"] = y[m & (v == 1)].mean() if (m & (v == 1)).any() else np.nan
        zr[c][f"{g}: вне"] = y[m & (v == 0)].mean() if (m & (v == 0)).any() else np.nan
        zr[c][f"{g}: доля в окне"] = (v[m] == 1).mean()
print("Доля target = 1 внутри и вне окна для лучших признаков-индикаторов:")
display(pd.DataFrame(zr).T)

## Проверка на моделях

In [ ]:
PARAMS = dict(objective="binary", learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=100, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50, random_state=SEED, n_jobs=-1, verbose=-1)
folds = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(tr, tr["slice"] + "_" + tr["target"].astype(int).astype(str)))


def fold_cells(p):
    return np.array([[fast_ap(y[b][sl[b] == q], p[b][sl[b] == q]) for q in slices] for _, b in folds])


def run_lgb(frame, features):
    p = np.zeros(len(frame))
    for a, b in folds:
        m = lgb.LGBMClassifier(**PARAMS).fit(frame[features].iloc[a], y[a], eval_set=[(frame[features].iloc[b], y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        p[b] = m.predict_proba(frame[features].iloc[b])[:, 1]
    return fold_cells(p)


def run_logreg(frame, features):
    p = np.zeros(len(frame))
    for a, b in folds:
        m = make_pipeline(SimpleImputer(strategy="median"), QuantileTransformer(n_quantiles=200, output_distribution="normal"), LogisticRegression(max_iter=2000))
        m.fit(frame[features].iloc[a], y[a])
        p[b] = m.predict_proba(frame[features].iloc[b])[:, 1]
    return fold_cells(p)


# до 40 лучших кандидатов по AUC 4+, без почти одинаковых (корреляция Спирмена 0.95 и выше)
order = [c for c in S[~S.index.str.startswith("[база]")].sort_values("AUC 4+ среднее", ascending=False).index[:200] if Xc[c].notna().mean() >= 0.05]
ranks = Xc[order].rank()
picked = []
for c in order:
    if len(picked) >= 40:
        break
    if all(abs(ranks[c].corr(ranks[p])) < 0.95 for p in picked):
        picked.append(c)
newc = [f"c{i}" for i in range(len(picked))]
frame = tr.copy()
frame[newc] = Xc[picked].to_numpy()

base_cells = run_lgb(frame, BASE)
new_cells = run_lgb(frame, BASE + newc)
d = (new_cells - base_cells).ravel()
print(f"LightGBM: 222 признака {base_cells.mean():.4f}; + {len(picked)} кандидатов {new_cells.mean():.4f}; "
      f"прирост {d.mean():+.4f} ± {d.std():.4f}, плюс в {int((d > 0).sum())} из {len(d)}")
SMALL = ["recency", "tenure", "n_days", "n_orders", "formula", "days_formula", "gap_mean", "tnext_mean", "tnext_mean_dist", "phit20_median"]
lr0, lr1 = run_logreg(frame, SMALL), run_logreg(frame, SMALL + newc)
print(f"Логистическая регрессия: 10 простых признаков {lr0.mean():.4f}; + кандидаты {lr1.mean():.4f}")
display(pd.Series(dict(zip(newc, picked))).to_frame("кандидат"))

## Определение target

In [ ]:
# На сдвинутых датах считаем метки по нескольким определениям и сравниваем с train по одним и тем же ячейкам
# «число дней с заказами x давность». Определение, которое ближе всех к train, подскажет, как устроен target.
dd = lines[[cid, "d"]].drop_duplicates().reset_index(drop=True)
dd_cut = dd[cid].map(info["cutoff"])
N_EDGES = [0, 1, 2, 3, 5, 9, 20, 10_000]
N_LABELS = ["1", "2", "3", "4-5", "6-9", "10-20", "21+"]
R_EDGES = [-1, 7, 14, 30, 60, 90, 150]
T_EDGES = list(range(-60, 151, 10))


def hist_stats(hd):
    g = hd.groupby(cid)["h"]
    st = pd.DataFrame({"recency": g.min(), "n_days": g.size()})
    st["tnext_mean"] = (g.max() - g.min()) / (st["n_days"] - 1).replace(0, np.nan) - st["recency"]
    return st


DEFS = {
    "A: любая покупка в окне": lambda x, W: (x > 14) & (x <= 14 + W),
    "C: окно от отсечения (0, W]": lambda x, W: (x > 0) & (x <= W),
    "D: только первые 2 месяца окна": lambda x, W: (x > 14) & (x <= 75),
    "G: окно на 14 дней позже": lambda x, W: (x > 28) & (x <= 28 + W),
}
parts = []
for k in range(4):
    b_off, b_w = {}, {}
    for c in sorted(info["cutoff"].unique()):
        c = pd.Timestamp(c)
        p_end = c - pd.DateOffset(months=k)
        p_date = p_end - pd.DateOffset(months=TARGET_MONTHS)
        p_cut = p_date - pd.Timedelta(days=GAP_DAYS)
        b_off[c], b_w[c] = (c - p_cut).days, (p_end - p_date).days
    off = dd_cut.map(pd.Series(b_off)).to_numpy()
    W = dd_cut.map(pd.Series(b_w)).to_numpy()
    x = off - dd["d"].to_numpy()                 # дней после сдвинутого отсечения; <= 0 — история
    hm = x <= 0
    st = hist_stats(pd.DataFrame({cid: dd[cid].to_numpy()[hm], "h": -x[hm]}))
    for name, f in DEFS.items():
        st[name] = pd.Series(f(x, W), index=dd.index).groupby(dd[cid]).any().reindex(st.index).fillna(False).astype(float)
    gap_buy = pd.Series((x > 0) & (x <= 14), index=dd.index).groupby(dd[cid]).any().reindex(st.index).fillna(False)
    st["B: первая покупка после отсечения — в окне"] = st["A: любая покупка в окне"] * (~gap_buy).astype(float)
    cnt = pd.Series(((x > 14) & (x <= 14 + W)).astype(int), index=dd.index).groupby(dd[cid]).sum().reindex(st.index).fillna(0)
    st["E: два и больше дней с покупками в окне"] = (cnt >= 2).astype(float)
    st["off"] = st.index.map(info["cutoff"]).map(pd.Series(b_off))
    parts.append(st[(st["recency"] + st["off"] <= 365) & (st["recency"] <= 150)].assign(shift=k))
SH = pd.concat(parts)
DEF_COLS = [c for c in SH.columns if c[:2] in ("A:", "B:", "C:", "D:", "E:", "G:")]

real = hist_stats(pd.DataFrame({cid: dd[cid], "h": dd["d"]})).reindex(tr_ids)
real["y"] = y
real = real[real["recency"] <= 150]


def cells(df, col):
    t = df.assign(nb=pd.cut(df["n_days"], N_EDGES, labels=N_LABELS), rb=pd.cut(df["recency"], R_EDGES))
    return t.groupby(["nb", "rb"], observed=False)[col].mean()


def profile(df, col):
    two = df[df["n_days"] == 2]
    return two.groupby(pd.cut(two["tnext_mean"], T_EDGES), observed=False)[col].mean()


ref = cells(real, "y")
ref_n = real.assign(nb=pd.cut(real["n_days"], N_EDGES, labels=N_LABELS), rb=pd.cut(real["recency"], R_EDGES)).groupby(["nb", "rb"], observed=False).size()
cmp, prof = {}, {"train": profile(real, "y")}
for c in DEF_COLS:
    cc = cells(SH, c)
    ok = (ref.notna() & cc.notna() & (ref_n >= 30)).to_numpy()
    cmp[c] = {"доля 1": SH[c].mean(), "ср. отклонение от train": float(np.average(np.abs(cc.to_numpy()[ok] - ref.to_numpy()[ok]), weights=ref_n.to_numpy()[ok])),
              "корреляция с train": float(pd.Series(cc.to_numpy()[ok]).corr(pd.Series(ref.to_numpy()[ok]), method="spearman"))}
    prof[c] = profile(SH, c)
cmp["train (давность <= 150)"] = {"доля 1": real["y"].mean()}
print("Насколько метки по разным определениям похожи на train (по ячейкам «число дней x давность»):")
display(pd.DataFrame(cmp).T)
print("Два дня с заказами: доля 1 по ожидаемому дню следующего заказа")
display(pd.DataFrame(prof).T)
best_def = min(DEF_COLS, key=lambda c: cmp[c]["ср. отклонение от train"])
print("Ближайшее к train определение:", best_def)
display(pd.concat({"train": ref.unstack(), best_def: cells(SH, best_def).unstack()}, axis=1))
print(f"Всего времени: {(time.time() - T0) / 60:.1f} мин")